# RAG Assignment: Baseline vs. Re-Ranking
## Part 1: Setup & Environment

In [1]:
# Install dependencies (run once)
# %pip install -q pypdf transformers torch faiss-cpu numpy ollama

In [2]:
import os, re, time
import numpy as np
import faiss
from pypdf import PdfReader
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

/home/abhishekg/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---- Configuration ----
PDF_PATH = "Privacy-Within-Metas-Integrity-Systems.pdf"   # chosen document: a 27-page technical/policy report
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder for Pipeline 1
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # cross-encoder for Pipeline 2
LLM_MODEL = "qwen3:8b"                                          # local LLM (Ollama) used for Answer 1 and Answer 2

CHUNK_SIZE = 200     # words per chunk
CHUNK_OVERLAP = 40   # words shared between consecutive chunks
TOP_K = 10           # candidates retrieved by the bi-encoder
TOP_N = 3            # chunks passed to the LLM

In [4]:
# ---- LLM (local model via Ollama) ----
# Takes the full prompt string and returns the model's answer as plain text.
# Temperature 0 (greedy), so Answer 1 and Answer 2 differ only in the retrieved chunks.
# Any failure raises; an error message is never returned as an answer.
import ollama

def generate(prompt: str) -> str:
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,  # Qwen3 is a hybrid reasoning model: thinking off gives the plain answer, same direct generation for both pipelines
        options={"temperature": 0, "seed": 0, "num_predict": 512, "num_ctx": 4096},
    )
    text = resp["message"]["content"].strip()
    if not text:
        raise RuntimeError("LLM returned an empty answer")
    return text

In [5]:
# ---- Load the PDF, keeping page numbers for citations ----
reader = PdfReader(PDF_PATH)
pages = [(i + 1, re.sub(r"\s+", " ", p.extract_text() or "").strip()) for i, p in enumerate(reader.pages)]
print(f"{len(pages)} pages, {sum(len(t.split()) for _, t in pages)} words")

27 pages, 8346 words


In [6]:
# ---- Divide the document into overlapping word-based chunks ----
# Both pipelines use this exact same list of chunks.
chunks = []  # each: {"id": int, "page": int, "text": str}
for page_no, text in pages:
    words = text.split()
    step = CHUNK_SIZE - CHUNK_OVERLAP
    for start in range(0, max(len(words), 1), step):
        piece = " ".join(words[start:start + CHUNK_SIZE])
        if len(piece.split()) >= 30:  # drop tiny fragments
            chunks.append({"id": len(chunks), "page": page_no, "text": piece})
print(f"{len(chunks)} chunks")
print(chunks[0])

59 chunks
{'id': 0, 'page': 2, 'text': 'What does privacy mean when data a person creates or data observed about them can be used to protect a broader community? This question is older than the Internet, 1 but feels particularly relevant for the modern day where many people spend a significant portion of their lives online. The Universal Declaration of Human Rights states that people have rights to both privacy and to safety, as well as numerous other rights like freedom of speech and association. 2 These values allow for autonomy, creation, and the many things that comprise living and growing in society. And while this charter was intended for and signed by governments, these values still offer important guideposts for what people desire, or even expect, out of their experiences with online companies. But when it comes to social media, discourse around privacy, speech, and safety can sometimes pull these values into different camps or pit them against each other — when in reality each

In [7]:
# ---- Load models ----
# sentence-transformers needs pyarrow, which is blocked by Application Control on this machine,
# so we use Hugging Face transformers directly (same models, same results).
class BiEncoder:
    """all-MiniLM-L6-v2: mean-pooled token embeddings, L2-normalised (so dot product = cosine)."""
    def __init__(self, name):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModel.from_pretrained(name).eval()

    @torch.no_grad()
    def encode(self, texts, batch_size=32):
        out = []
        for i in range(0, len(texts), batch_size):
            b = self.tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=256, return_tensors="pt")
            h = self.model(**b).last_hidden_state
            mask = b["attention_mask"].unsqueeze(-1).float()
            emb = (h * mask).sum(1) / mask.sum(1)
            out.append(torch.nn.functional.normalize(emb, dim=1))
        return torch.cat(out).numpy().astype("float32")

class CrossEncoder:
    """ms-marco cross-encoder: scores each (query, passage) pair jointly."""
    def __init__(self, name):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(name).eval()

    @torch.no_grad()
    def predict(self, pairs):
        b = self.tok([q for q, _ in pairs], [p for _, p in pairs], padding=True, truncation=True, max_length=512, return_tensors="pt")
        return self.model(**b).logits.squeeze(-1).numpy()

bi_encoder = BiEncoder(BI_ENCODER_NAME)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21245.86it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15687.18it/s]


BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Shared components
Both pipelines run over the *same* chunk list. The vector store is a FAISS inner-product index; because the
bi-encoder returns L2-normalised vectors, inner product equals cosine similarity.

In [8]:
# ---- Index all chunks in a vector store (bi-encoder embeddings) ----
chunk_texts = [c["text"] for c in chunks]
chunk_embs = bi_encoder.encode(chunk_texts)           # shape (n_chunks, 384), normalised
index = faiss.IndexFlatIP(chunk_embs.shape[1])        # exact inner-product search
index.add(chunk_embs)
print("indexed", index.ntotal, "chunks")

indexed 59 chunks


In [9]:
# ---- Identical prompt for both pipelines, so only the retrieved chunks differ ----
PROMPT = """You are answering questions about a single document using ONLY the context below.
If the context does not contain the answer, say "The provided context does not answer this."

Context:
{context}

Question: {question}

Answer concisely:"""

def build_prompt(question, chunk_ids):
    context = "\n\n".join(f"[Chunk {i}, page {chunks[i]['page']}] {chunks[i]['text']}" for i in chunk_ids)
    return PROMPT.format(context=context, question=question)

def generate_with_retry(prompt, tries=3):
    # retry transient failures; a stub / config error is raised straight away, and
    # the final failure is raised too, so an error can never be stored as an answer
    for t in range(tries):
        try:
            return generate(prompt)
        except NotImplementedError:
            raise
        except Exception:
            if t == tries - 1:
                raise
            time.sleep(2)

## Pipeline 1: Baseline Retrieval
1. Embed the query with the bi-encoder.
2. Take the top 10 chunks by cosine similarity from the vector store.
3. Give the top 3 directly to the LLM to produce **Answer 1**.

In [10]:
def baseline_retrieve(query, k=TOP_K):
    """Bi-encoder retrieval. Returns [(chunk_id, cosine_score), ...] best first."""
    q = bi_encoder.encode([query])
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

def pipeline1(query):
    cands = baseline_retrieve(query)                   # top 10
    top3 = [cid for cid, _ in cands[:TOP_N]]           # taken directly, no re-ordering
    return cands, top3, generate_with_retry(build_prompt(query, top3))

## Pipeline 2: Advanced Re-Ranking
1. Take the same top 10 candidates from the baseline retrieval.
2. Score all 10 (query, chunk) pairs with the cross-encoder.
3. Re-sort by cross-encoder score and keep the top 3.
4. Give them to the LLM with the identical prompt to produce **Answer 2**.

In [11]:
def rerank(query, cands):
    """Cross-encoder re-ranking of the baseline candidates. Returns [(chunk_id, ce_score), ...] best first."""
    scores = cross_encoder.predict([(query, chunks[cid]["text"]) for cid, _ in cands])
    order = np.argsort(-scores)
    return [(cands[i][0], float(scores[i])) for i in order]

def pipeline2(query, cands=None):
    cands = cands or baseline_retrieve(query)          # same top 10 as pipeline 1
    reranked = rerank(query, cands)
    top3 = [cid for cid, _ in reranked[:TOP_N]]
    return reranked, top3, generate_with_retry(build_prompt(query, top3))

## Failure Mode Analysis (Stress Testing)
Each test query has a `gold` string that identifies the chunk(s) containing the true answer. That lets us report
the gold chunk's rank under the baseline vs. after re-ranking automatically, and then read the two LLM answers.

| # | Category | Why it should be hard for a bi-encoder |
|---|----------|----------------------------------------|
| 1 | Exceptions & corner cases | The general rule (data deleted) and the exception (child-safety accounts) sit in the same chunk, and other chunks also talk about retention. |
| 2 | Buried chunk (rank 10) | Query wording ("delete", "profile", "photos") overlaps many chunks about data and accounts. |
| 3 | Buried chunk, specific detail | A short, factual contact detail hidden in a chunk of mostly generic closing text. |
| 4 | Negation & constraints | "Helps spot a hacked account but is *not* useful for judging nudity": embeddings blur negation, so nudity-detection chunks outrank the passage that says which signals matter little for it. |
| 5 | Keyword trap / distractor | "hate speech", "remove" and "Meta" match the hate-speech case-study chunks, which describe removal but give no figure; the number sits in an endnote about the enforcement report. |
| 6 | Control: reranking does not help | Included deliberately to show the cross-encoder can also lose. |

In [12]:
FAILURE_QUERIES = [
    {"cat": "Exceptions & corner cases",
     "q": "Does Meta retain account data when a user deletes their account?",
     "gold": "typically retain"},
    {"cat": "Buried chunk (rank 10)",
     "q": "What happens to a user's profile and photos when they delete their Facebook account?",
     "gold": "permanently delete your profile"},
    {"cat": "Buried chunk, specific detail",
     "q": "What is the deadline to send feedback on the paper and to which address?",
     "gold": "privacy4integrity"},
    {"cat": "Negation & constraints",
     "q": "Which account signal helps spot a hacked account but is not useful for deciding whether a photo is nudity?",
     "gold": "little relevance for determining if a picture is nudity"},
    {"cat": "Keyword trap / distractor",
     "q": "How much hate speech did Meta remove in late 2021?",
     "gold": "17.4 million"},
    {"cat": "Control: reranking does not help",
     "q": "Which privacy principle says data should be kept only as long as required, and what is the retention period for hate speech?",
     "gold": "two years"},
]

def gold_ids(gold):
    return [c["id"] for c in chunks if gold in c["text"]]

def rank_of(ids, order):
    """Best (smallest) 1-based rank of any gold chunk in an ordered list of chunk ids, or None."""
    ranks = [order.index(i) + 1 for i in ids if i in order]
    return min(ranks) if ranks else None

In [13]:
results = []
for item in FAILURE_QUERIES:
    q = item["q"]
    cands, top3_a, ans1 = pipeline1(q)
    reranked, top3_b, ans2 = pipeline2(q, cands)
    g = gold_ids(item["gold"])
    results.append({**item, "gold_ids": g,
                    "base_rank": rank_of(g, [c for c, _ in cands]),
                    "rerank_rank": rank_of(g, [c for c, _ in reranked]),
                    "top3_base": top3_a, "top3_rerank": top3_b,
                    "ans1": ans1, "ans2": ans2})
    time.sleep(2)  # be gentle with the API rate limit
print("done")

done


### Rank shift table

In [14]:
print(f"{'#':<3}{'Category':<34}{'Gold chunk(s)':<15}{'Baseline rank':<15}{'Re-ranked rank':<15}{'In top3 (base -> rerank)'}")
for n, r in enumerate(results, 1):
    in_a = r["base_rank"] is not None and r["base_rank"] <= TOP_N
    in_b = r["rerank_rank"] is not None and r["rerank_rank"] <= TOP_N
    print(f"{n:<3}{r['cat']:<34}{str(r['gold_ids']):<15}{str(r['base_rank']):<15}{str(r['rerank_rank']):<15}{in_a} -> {in_b}")

#  Category                          Gold chunk(s)  Baseline rank  Re-ranked rank In top3 (base -> rerank)
1  Exceptions & corner cases         [30]           7              1              False -> True
2  Buried chunk (rank 10)            [30]           10             1              False -> True
3  Buried chunk, specific detail     [43]           7              1              False -> True
4  Negation & constraints            [6]            7              1              False -> True
5  Keyword trap / distractor         [48]           6              1              False -> True
6  Control: reranking does not help  [22]           2              4              True -> False


### Answer 1 (baseline) vs. Answer 2 (re-ranked)

In [15]:
for n, r in enumerate(results, 1):
    print("=" * 100)
    print(f"[{n}] {r['cat']}\nQ: {r['q']}")
    print(f"Gold chunk(s): {r['gold_ids']}  (pages {sorted({chunks[i]['page'] for i in r['gold_ids']})})")
    print(f"Top-3 baseline : {r['top3_base']}   | Top-3 re-ranked: {r['top3_rerank']}")
    print(f"\nAnswer 1 (baseline):\n{r['ans1']}")
    print(f"\nAnswer 2 (re-ranked):\n{r['ans2']}")

[1] Exceptions & corner cases
Q: Does Meta retain account data when a user deletes their account?
Gold chunk(s): [30]  (pages [14])
Top-3 baseline : [31, 29, 14]   | Top-3 re-ranked: [30, 31, 14]

Answer 1 (baseline):
The provided context does not answer this.

Answer 2 (re-ranked):
Yes, Meta retains account data from child safety-disabled accounts to proactively prevent known violators from continuing their behavior through new accounts, but does not typically retain this type of information from deleted, non-violating accounts.
[2] Buried chunk (rank 10)
Q: What happens to a user's profile and photos when they delete their Facebook account?
Gold chunk(s): [30]  (pages [14])
Top-3 baseline : [32, 56, 37]   | Top-3 re-ranked: [30, 37, 32]

Answer 1 (baseline):
The provided context does not answer this.

Answer 2 (re-ranked):
When a user deletes their Facebook account, their profile, photos, and everything they have added to the account are permanently deleted.
[3] Buried chunk, specifi

## Latency Profiling
Retrieval-only timings (no LLM), median over repeated runs after a warm-up. LLM time is measured separately because
it dominates both pipelines.

In [16]:
import statistics

def median_ms(fn, runs=20):
    fn()  # warm-up
    times = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); times.append((time.perf_counter() - t0) * 1000)
    return statistics.median(times)

rows = []
for item in FAILURE_QUERIES:
    q = item["q"]
    cands = baseline_retrieve(q)
    t_base = median_ms(lambda: baseline_retrieve(q))            # embed query + FAISS top-10
    t_rr = median_ms(lambda: rerank(q, cands))                  # cross-encoder on 10 pairs
    rows.append((t_base, t_rr))

print(f"{'Query #':<9}{'Baseline (ms)':<16}{'Re-rank step (ms)':<20}{'Pipeline 2 total (ms)':<23}{'Slowdown'}")
for n, (b, r) in enumerate(rows, 1):
    print(f"{n:<9}{b:<16.1f}{r:<20.1f}{b + r:<23.1f}{(b + r) / b:.1f}x")
mb = statistics.mean(b for b, _ in rows); mr = statistics.mean(r for _, r in rows)
print(f"{'Mean':<9}{mb:<16.1f}{mr:<20.1f}{mb + mr:<23.1f}{(mb + mr) / mb:.1f}x")

# LLM generation time (prompt size is about the same in both pipelines)
q = FAILURE_QUERIES[0]["q"]
t0 = time.perf_counter(); generate_with_retry(build_prompt(q, results[0]["top3_base"])); print(f"\nOne LLM call: {(time.perf_counter() - t0) * 1000:.0f} ms")

Query #  Baseline (ms)   Re-rank step (ms)   Pipeline 2 total (ms)  Slowdown
1        6.9             683.1               690.0                  100.4x
2        5.1             293.9               299.1                  58.1x
3        6.8             680.9               687.8                  100.6x
4        7.4             594.3               601.7                  81.5x
5        7.4             522.4               529.8                  72.1x
6        6.6             541.0               547.6                  83.0x
Mean     6.7             552.6               559.3                  83.5x



One LLM call: 636 ms
